In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW_DIR = Path("../data/raw/archive")
PROCESSED_DIR = Path("../data/processed")
REPORTS_DIR = Path("../reports")
FIG_DIR = REPORTS_DIR / "figures"

RAW_DIR.exists(), PROCESSED_DIR.exists(), FIG_DIR.exists()

(True, True, True)

In [2]:
clean_path = RAW_DIR / "budgetwise_finance_dataset.csv"
dirty_path = RAW_DIR / "budgetwise_synthetic_dirty.csv"

df_clean = pd.read_csv(clean_path)
df_dirty = pd.read_csv(dirty_path)

df_clean.shape, df_dirty.shape

((15900, 9), (15836, 9))

In [3]:
def quick_profile(df, name):
    print(f"--- {name} ---")
    print("shape:", df.shape)
    display(df.head(3))
    display(df.dtypes)
    display(df.isna().mean().sort_values(ascending=False).head(15))

quick_profile(df_clean, "CLEAN")
quick_profile(df_dirty, "DIRTY")

--- CLEAN ---
shape: (15900, 9)


,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
0,T4999,U018,2023-04-25,Expense,Educaton,3888,card,Ahmedabad,Movie tickets
1,T12828,U133,08/05/2022,Expense,rent,649,NaN,Hyderabad,asdfgh
2,T7403,U091,31-12-23,Income,Freelance,13239,Csh,BAN,Books


transaction_id      str
user_id             str
date                str
transaction_type    str
category            str
amount              str
payment_mode        str
location            str
notes               str
dtype: object

notes               0.177421
location            0.079371
payment_mode        0.050818
date                0.030566
amount              0.018302
category            0.017925
transaction_id      0.000000
user_id             0.000000
transaction_type    0.000000
dtype: float64

--- DIRTY ---
shape: (15836, 9)


,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
0,T03512,U039,December 22 2021,Expense,Rent,998,Cash,Pune,Paid electricity bill
1,T03261,U179,03/24/2022,Expense,Food,$143,Card,Delhi,Grocery shopping
2,T04316,U143,October 18 2022,Expense,Rent,149,Cash,Bengaluru,NaN


transaction_id      str
user_id             str
date                str
transaction_type    str
category            str
amount              str
payment_mode        str
location            str
notes               str
dtype: object

notes               0.096868
location            0.045592
payment_mode        0.031763
date                0.021723
amount              0.011240
category            0.009977
transaction_id      0.000000
user_id             0.000000
transaction_type    0.000000
dtype: float64

In [4]:
clean_cols = set(df_clean.columns)
dirty_cols = set(df_dirty.columns)

print("Only in CLEAN:", sorted(clean_cols - dirty_cols))
print("Only in DIRTY:", sorted(dirty_cols - clean_cols))
print("Common:", len(clean_cols & dirty_cols))

Only in CLEAN: []
Only in DIRTY: []
Common: 9


In [5]:
df_dirty.columns.tolist()

['transaction_id',
 'user_id',
 'date',
 'transaction_type',
 'category',
 'amount',
 'payment_mode',
 'location',
 'notes']

In [8]:
df = df_dirty.copy()

profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_pct": df.isna().mean().round(4),
    "n_unique": df.nunique(dropna=True),
    "sample_values": [df[c].dropna().astype(str).head(3).tolist() for c in df.columns],
}).sort_values("missing_pct", ascending=False)

profile

,dtype,missing_pct,n_unique,sample_values
notes,str,0.0969,988,"[Paid electricity bill , Grocery shopping, Pai..."
location,str,0.0456,30,"[Pune, Delhi, Bengaluru]"
payment_mode,str,0.0318,62,"[Cash, Card, Cash]"
date,str,0.0217,5319,"[December 22 2021, 03/24/2022, October 18 2022]"
amount,str,0.0112,6279,"[998, $143, 149]"
category,str,0.0100,212,"[Rent, Food, Rent]"
transaction_id,str,0.0000,14858,"[T03512, T03261, T04316]"
user_id,str,0.0000,192,"[U039, U179, U143]"
transaction_type,str,0.0000,2,"[Expense, Expense, Expense]"


In [9]:
dt = pd.to_datetime(df_dirty["date"], errors="coerce")
date_fail_rate = dt.isna().mean()
date_fail_count = dt.isna().sum()

date_fail_rate, date_fail_count

(np.float64(0.8516039403889871), np.int64(13486))

In [10]:
df_dirty.loc[dt.isna(), "date"].value_counts().head(30)

date
2020-08-29    12
10/13/2020    11
2021-04-16    10
04-08-21      10
2022-01-24    10
2021-04-03    10
11/08/2020    10
2019-11-11    10
10/04/2022     9
2022-02-09     9
2022-03-15     9
06/25/2019     9
2021-02-15     9
07/08/2021     9
2020-02-24     9
07/05/2022     9
07/27/2019     9
11/28/2022     9
2022-08-06     9
08/11/2020     9
12-09-20       9
2019-09-16     9
04/08/2020     9
2019-09-07     9
30-10-22       9
08/23/2021     9
06/23/2022     9
10/24/2019     8
06/02/2020     8
2019-03-21     8
Name: count, dtype: int64

In [11]:
dup_id_count = df_dirty["transaction_id"].duplicated().sum()
dup_id_rate = df_dirty["transaction_id"].duplicated().mean()

dup_id_rate, dup_id_count

(np.float64(0.06175801970194494), np.int64(978))

In [12]:
df_dirty[df_dirty["transaction_id"].duplicated(keep=False)]\
    .sort_values("transaction_id")\
    .head(20)

,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
968,T00020,U182,11/02/2021,Expense,Food,250,Card,NaN,NaN
6992,T00020,U182,11/02/2021,Expense,Food,250,Card,NaN,NaN
4936,T00028,U127,14-01-21,Expense,Entertainment,"3,352",UPI,Jaipur,Grocery shopping
13263,T00028,U127,14-01-21,Expense,Entertainment,"3,352",UPI,Jaipur,Grocery shopping
6303,T00048,U175,September 13 2021,Expense,Utilities,"$1,167",Bank Transfer,Lucknow,Doctor visit
10833,T00048,U175,September 13 2021,Expense,Utilities,"$1,167",Bank Transfer,Lucknow,Doctor visit
5482,T00058,U136,12-06-21,Expense,Rent,26,Cash,Mumbai,Salary
9870,T00058,U034,2020-04-10,Expense,Ret,NaN,Cash,Delhi,ZwWiyLgqiyCxsx
6463,T00089,U125,2022-07-20,Expense,Rent,158,UPI,Lucknow,Monthly rent
14637,T00089,U125,2022-07-20,Expense,Rent,158,UPI,Lucknow,Monthly rent
